In [51]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from pykalman import KalmanFilter
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

In [43]:
# 设置文件夹路径
json_folder = "../data/video_data/"
output_folder = "../data/trajectory_plots/Kalman_Filter"

In [44]:
# 确保输出目录存在
os.makedirs(output_folder, exist_ok=True)

In [45]:
# 单位换算因子(英尺转米)
FEET_TO_METERS = 0.3048

# 设定绘图范围
SCREEN_X_MIN, SCREEN_X_MAX = 0, 10
SCREEN_Y_MIN, SCREEN_Y_MAX = -50, 300

In [46]:
# 获取所有 JSON 文件
json_files = [f for f in os.listdir(json_folder) if f.endswith(".json")]

In [47]:
# 卡尔曼滤波器

def kalman_smoothing(y):
    """ 使用卡尔曼滤波器对轨迹进行平滑 """
    kf = KalmanFilter(initial_state_mean=[y[0], 0], n_dim_obs=1, n_dim_state=2)
    kf.transition_matrices = [[1, 1], [0, 1]]  # 状态转移矩阵
    kf.observation_matrices = [[1, 0]]  # 观测矩阵
    smoothed_state_means, _ = kf.smooth(y.reshape(-1, 1))
    return smoothed_state_means[:, 0]  # 仅返回平滑后的轨迹

In [58]:
def gpr_smoothing(x, y):
    """ 使用高斯过程回归（GPR）进行轨迹平滑 """
    x = x.reshape(-1, 1)  # GPR 需要二维输入

    # 归一化数据
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    x_scaled = scaler_x.fit_transform(x)
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

    # **调整核函数参数**
    kernel = C(1.0, (1e-2, 1e3)) * RBF(length_scale=10, length_scale_bounds=(1, 100))
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=30)
    gpr.fit(x_scaled, y_scaled)  # 拟合归一化数据

    # 预测并反归一化
    y_pred_scaled = gpr.predict(x_scaled)
    return scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()


In [59]:
# 遍历每个 JSON 文件并处理
for json_file in json_files:
    try:
        json_path = os.path.join(json_folder, json_file)
        
        # 读取 JSON 文件
        with open(json_path, "r") as f:
            data = json.load(f)
        
        # 提取车辆数据
        vehicles = data.get("vehicleDataList", [])
        vehicle_paths = []

        for vehicle in vehicles:
            vehicle_id = vehicle["vehicleId"]
            v_class = vehicle.get("vClass", "N/A")

            if "path" in vehicle and "frame" in vehicle:
                for i, point in enumerate(vehicle["path"]):
                    if i < len(vehicle["frame"]):
                        frame = vehicle["frame"][i]
                        vehicle_paths.append({
                            "vehicleId": vehicle_id,
                            "vClass": v_class,
                            "localX": point["localX"] * FEET_TO_METERS,  # X 轴转换为米
                            "localY": point["localY"] * FEET_TO_METERS,  # Y 轴转换为米
                            "laneId": frame["laneId"],
                            "velocity": frame["velocity"] * FEET_TO_METERS  # 速度转换为米/秒
                        })

        if not vehicle_paths:
            print(f"⚠ 跳过 {json_file}，没有轨迹数据")
            continue

        df_paths = pd.DataFrame(vehicle_paths)
        unique_vehicles = df_paths["vehicleId"].unique()

        # 获取数据的 Y 轴最小、最大范围
        data_y_min, data_y_max = df_paths["localY"].min(), df_paths["localY"].max()

        # 计算 Y 轴缩放因子
        scale_y = (SCREEN_Y_MAX - SCREEN_Y_MIN) / (data_y_max - data_y_min)
        scale_x = 0.3  # 设定 X 轴缩放倍数

        # 应用缩放
        df_paths["localX"] = df_paths["localX"] * scale_x
        df_paths["localY"] = (df_paths["localY"] - data_y_min) * scale_y + SCREEN_Y_MIN

        # 先使用卡尔曼滤波进行平滑处理
        for vid in unique_vehicles:
            mask = df_paths["vehicleId"] == vid
            if len(df_paths[mask]) > 3:  # 至少需要3个点才能进行滤波
                df_paths.loc[mask, "localX"] = kalman_smoothing(df_paths.loc[mask, "localX"].values)
                df_paths.loc[mask, "localY"] = kalman_smoothing(df_paths.loc[mask, "localY"].values)
        
        # 再使用 GPR 进行更精细的平滑处理
        for vid in unique_vehicles:
            mask = df_paths["vehicleId"] == vid
            if len(df_paths[mask]) > 3:
                df_paths.loc[mask, "localX"] = gpr_smoothing(df_paths.loc[mask, "localX"].index.values, df_paths.loc[mask, "localX"].values)
                df_paths.loc[mask, "localY"] = gpr_smoothing(df_paths.loc[mask, "localY"].index.values, df_paths.loc[mask, "localY"].values)

        # 绘制轨迹图
        plt.figure(figsize=(5, 15), facecolor="gray")
        plt.xlim(SCREEN_X_MIN, SCREEN_X_MAX)
        plt.ylim(SCREEN_Y_MIN, SCREEN_Y_MAX)
        plt.xlabel("X Position (m)")
        plt.ylabel("Y Position (m)")
        plt.title(f"Vehicle Trajectories - {json_file}")
        plt.grid(True, linestyle='--', alpha=0.5)

        # 绘制所有车辆的轨迹，调整线条宽度
        for vid in unique_vehicles:
            vehicle_data = df_paths[df_paths["vehicleId"] == vid]
            plt.plot(vehicle_data["localX"], vehicle_data["localY"], linewidth=0.5, label=f"Vehicle {vid}")
        
        plt.legend()
        output_filename = os.path.join(output_folder, json_file.replace(".json", ".png"))
        plt.savefig(output_filename)
        plt.close()
        print(f"✅ 轨迹图 {output_filename} 生成成功！")
    
    except Exception as e:
        print(f"❌ 处理 {json_file} 时出错: {e}")

D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavio

✅ 轨迹图 ../data/trajectory_plots/Kalman_Filter\i-80~1.png 生成成功！


D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavio

✅ 轨迹图 ../data/trajectory_plots/Kalman_Filter\i-80~10.png 生成成功！


D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavio

...

D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavio

✅ 轨迹图 ../data/trajectory_plots/Kalman_Filter\i-80~8.png 生成成功！


D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavio

✅ 轨迹图 ../data/trajectory_plots/Kalman_Filter\i-80~9.png 生成成功！


D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
D:\Anaconda\envs\vehicle_behavior_analysis\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
